<a href="https://colab.research.google.com/github/MuhammadRabees/Agentic-RAG/blob/main/Agentic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-groq
!pip install -q faiss-cpu tiktoken pypdf pandas langgraph
!pip install -q sentence-transformers
!pip install -q streamlit pyngrok

In [ ]:
!pip install -q --upgrade langchain-groq langchain-core langgraph

In [3]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]   = userdata.get("RAG_API_KEY")


In [4]:
import pandas as pd
import ast
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_csv_tickets(csv_path: str) -> list[Document]:
    """
    Loads the Kaggle IT ticket dataset with columns:
    Body, Department, Priority, Tags
    Each row becomes one LangChain Document.
    Body > 500 chars → split into sub-chunks but metadata stays attached.
    """
    df = pd.read_csv(csv_path)

    print(f" Dataset shape: {df.shape}")
    print(f" Columns found: {list(df.columns)}")

    df = df.dropna(subset=["Body"])
    df["Department"] = df.get("Department", pd.Series(["General"] * len(df))).fillna("General")
    df["Priority"]   = df.get("Priority",   pd.Series(["Medium"]  * len(df))).fillna("Medium")
    df["Tags"]       = df.get("Tags",        pd.Series(["[]"]      * len(df))).fillna("[]")

    print(f" Clean rows after dropping empty Body: {len(df)}")

    ESCALATE_PRIORITIES = {"critical", "urgent", "high"}

    body_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=80,
        separators=["\n\n", "\n", ".", " "]
    )

    docs = []
    split_count = 0

    for idx, row in df.iterrows():

        # Parse Tags
        tags = row["Tags"]
        if isinstance(tags, str):
            try:
                tags = ast.literal_eval(tags)
            except Exception:
                tags = [tags]
        tags_str = ", ".join(tags) if isinstance(tags, list) else str(tags)

        should_escalate = str(row["Priority"]).strip().lower() in ESCALATE_PRIORITIES

        meta = {
            "source":          "kaggle_it_tickets",
            "department":      str(row["Department"]).strip(),
            "priority":        str(row["Priority"]).strip(),
            "tags":            tags_str,
            "should_escalate": should_escalate,
            "ticket_index":    int(idx),
        }

        body_text = str(row["Body"]).strip()

        if len(body_text) > 800:
            # Split the long body into pieces
            body_parts = body_splitter.split_text(body_text)
            split_count += 1
            for part_num, part in enumerate(body_parts):
                # Each chunk keeps Department/Priority/Tags attached
                content = (
                    f"IT Support Ticket\n"
                    f"───────────────\n"
                    f"Issue: {part}\n"
                    f"Department: {row['Department']}\n"
                    f"Priority: {row['Priority']}\n"
                    f"Tags: {tags_str}"
                )
                chunk_meta = {**meta, "chunk_part": part_num + 1,
                              "total_parts": len(body_parts)}
                docs.append(Document(page_content=content, metadata=chunk_meta))
        else:
            # Body fits in one chunk — keep row fully intact
            content = (
                f"IT Support Ticket\n"
                f"───────────────\n"
                f"Issue: {body_text}\n"
                f"Department: {row['Department']}\n"
                f"Priority: {row['Priority']}\n"
                f"Tags: {tags_str}"
            )
            meta["chunk_part"]   = 1
            meta["total_parts"]  = 1
            docs.append(Document(page_content=content, metadata=meta))

    print(f" Total documents/chunks created: {len(docs)}")
    print(f"  Rows that were split (Body > 800 chars): {split_count}")
    print(f" Rows kept intact (Body ≤ 800 chars): {len(df) - split_count}")

    print("\n Sample document (first ticket):")
    print(docs[0].page_content)
    print(f"\n  Metadata: {docs[0].metadata}")

    return docs


def show_dataset_stats(csv_path: str):
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["Body"])

    print("=" * 45)
    print(" DATASET OVERVIEW")
    print("=" * 45)
    print(f"Total tickets:    {len(df)}")
    print(f"\n Department breakdown:")
    print(df["Department"].value_counts().to_string())
    print(f"\n Priority breakdown:")
    print(df["Priority"].value_counts().to_string())
    print(f"\n Avg Body length (chars): {df['Body'].str.len().mean():.0f}")
    print(f" Max Body length (chars): {df['Body'].str.len().max()}")

    over_800 = (df['Body'].str.len() > 800).sum()
    print(f" Rows with Body > 800 chars: {over_800}")
    print("=" * 45)


CSV_PATH = "/content/IT Support Ticket Data.csv"

show_dataset_stats(CSV_PATH)

all_docs = load_csv_tickets(CSV_PATH)
chunks = all_docs

 DATASET OVERVIEW
Total tickets:    29650

 Department breakdown:
Department
Technical Support                  8617
Product Support                    5538
Customer Service                   4482
IT Support                         3500
Billing and Payments               3017
Returns and Exchanges              1467
Service Outages and Maintenance    1157
Sales and Pre-Sales                 885
Human Resources                     568
General Inquiry                     419

 Priority breakdown:
Priority
medium    12126
high      11511
low        6013

 Avg Body length (chars): 385
 Max Body length (chars): 2422
 Rows with Body > 800 chars: 1022
 Dataset shape: (29651, 5)
 Columns found: ['Unnamed: 0', 'Body', 'Department', 'Priority', 'Tags']
 Clean rows after dropping empty Body: 29650
 Total documents/chunks created: 30758
  Rows that were split (Body > 800 chars): 1022
 Rows kept intact (Body ≤ 800 chars): 28628

 Sample document (first ticket):
IT Support Ticket
───────────────
Issu

In [ ]:

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("⏳ Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

print("⏳ Embedding 30,758 chunks... (5-8 mins on Colab CPU)")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("/content/IT_helpdesk_faiss_index")
print(" Vector store saved!")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
import os

# ── Load vector store ─────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
vectorstore = FAISS.load_local(
    "/content/IT_helpdesk_faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✅ Vector store loaded!")

# ── Tool 1 ────────────────────────────────────
@tool
def search_knowledge_base(query: str) -> str:
    """Search the IT knowledge base for relevant solutions to any IT issue or question."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant documents found in the knowledge base."
    results = []
    for i, doc in enumerate(docs, 1):
        dept     = doc.metadata.get("department", "Unknown")
        priority = doc.metadata.get("priority", "Unknown")
        tags     = doc.metadata.get("tags", "")
        results.append(
            f"[Result {i} | Dept: {dept} | Priority: {priority} | Tags: {tags}]\n"
            f"{doc.page_content}"
        )
    return "\n\n---\n\n".join(results)

# ── Tool 2 ────────────────────────────────────
@tool
def search_past_tickets(issue_description: str) -> str:
    """Search past resolved IT tickets for similar issues. Use when user describes a specific error or recurring problem."""
    docs = retriever.invoke(issue_description)
    if not docs:
        return "No matching past tickets found."
    return "\n\n---\n\n".join(doc.page_content for doc in docs[:3])

# ── Tool 3 ────────────────────────────────────
@tool
def escalate_issue(issue_summary: str) -> str:
    """Escalate to a human IT agent when problem cannot be resolved or user asks for human help."""
    return (
        f" ESCALATION TRIGGERED\n"
        f"Issue: {issue_summary}\n"
        f"Ticket #ESC-{hash(issue_summary) % 10000:04d} created.\n"
        f"Human IT agent will respond within 2 business hours."
    )

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ.get("GROQ_API_KEY")
)

SYSTEM_PROMPT = """You are an expert IT Helpdesk Assistant for a company.
Help employees resolve IT issues quickly and accurately.

RULES:
1. Always use search_knowledge_base first for any IT issue.
2. Also use search_past_tickets if issue sounds recurring.
3. Provide step-by-step resolution instructions.
4. Mention which department/tags the answer came from.
5. NEVER auto-escalate — always ask user permission first.
6. Only call escalate_issue tool AFTER user confirms with "yes".
7. Before escalating, ask for user's email address.
8. Never make up solutions — only use what tools return.

ESCALATION FLOW — follow this exactly:
- If no solution found → ask:
  "I couldn't find a solution. Should I escalate this to a human IT agent? (yes/no)"
- If user says yes → ask:
  "Please provide your email address so the IT agent can contact you."
- After getting email → call escalate_issue tool with issue + email included.
- If user says no → suggest they try again with more details.

FORMAT:
 Issue understood: [restate briefly]
 Source: [department + tags]
 Steps to resolve: [numbered steps]
 If this doesn't work: Should I escalate this to a human IT agent? (yes/no)
"""

# ── Tools list PEHLE, agent BAAD MEIN ─────────
tools = [search_knowledge_base, search_past_tickets, escalate_issue]

# ── Agent — NO bind_tools ──────────────────────
agent = create_react_agent(
    llm,
    tools,
    prompt=SystemMessage(content=SYSTEM_PROMPT)
)
print("✅ Agent ready!")

In [7]:
import time

def ask_agent(question: str) -> str:
    """Run a question through the agent and return final answer."""
    result = agent.invoke({"messages": [("human", question)]})
    for msg in reversed(result["messages"]):
        if hasattr(msg, "type") and msg.type == "ai" and msg.content:

            content = msg.content
            if isinstance(content, list):

                text_parts = [
                    item["text"] for item in content
                    if isinstance(item, dict) and item.get("type") == "text"
                ]
                return "\n".join(text_parts)
            return content
    return "No answer generated."

questions = [
    "My VPN keeps disconnecting every 10 minutes. How do I fix it?"
]

for q in questions:
    print(f"\n{'='*55}")
    print(f" USER: {q}")
    print(f"{'='*55}")
    print(f" AGENT:\n{ask_agent(q)}")
    print(" Waiting....")
    time.sleep(15)


 USER: My VPN keeps disconnecting every 10 minutes. How do I fix it?
 AGENT:
Issue understood: Your VPN keeps disconnecting every 10 minutes.
 Source: Technical Support | Network, Disruption, Hardware, VPN, Router
 Steps to resolve:
1. Check the VPN-router and local network's stability.
2. Reboot the devices and inspect the cables.
3. If the issue persists, try restarting the router and checking the internet speed.
 If this doesn't work: Should I escalate this to a human IT agent? (yes/no)
 Waiting....


In [8]:
import os
from google.colab import userdata

# Pehle key fetch karo Colab Secrets se
groq_key = userdata.get("RAG_API_KEY")

# Verify karo ke key mili
print(f"Key found: {groq_key[:8]}...")

STREAMLIT_APP = '''
import os
import streamlit as st
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent

os.environ["GROQ_API_KEY"] = "''' + groq_key + '''"

@st.cache_resource
def load_agent():
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"}
    )
    vectorstore = FAISS.load_local(
        "/content/IT_helpdesk_faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    @tool
    def search_knowledge_base(query: str) -> str:
        """Search the IT knowledge base for relevant solutions to any IT issue or question."""
        docs = retriever.invoke(query)
        if not docs:
            return "No relevant documents found."
        results = []
        for i, doc in enumerate(docs, 1):
            dept     = doc.metadata.get("department", "Unknown")
            priority = doc.metadata.get("priority", "Unknown")
            tags     = doc.metadata.get("tags", "")
            results.append(
                f"[Result {i} | Dept: {dept} | Priority: {priority} | Tags: {tags}]\\n"
                f"{doc.page_content}"
            )
        return "\\n\\n---\\n\\n".join(results)

    @tool
    def search_past_tickets(issue_description: str) -> str:
        """Search past resolved IT tickets for similar issues."""
        docs = retriever.invoke(issue_description)
        if not docs:
            return "No matching past tickets found."
        return "\\n\\n---\\n\\n".join(doc.page_content for doc in docs[:3])

    @tool
    def escalate_issue(issue_summary: str) -> str:
        """Escalate to a human IT agent — only call this after user confirms yes and provides email."""
        return (
            f" ESCALATION TRIGGERED\\n"
            f"Issue: {issue_summary}\\n"
            f"Ticket #ESC-{hash(issue_summary) % 10000:04d} created.\\n"
            f"Human IT agent will respond within 2 business hours."
        )

    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0,
        api_key=os.environ.get("GROQ_API_KEY")
    )

    SYSTEM_PROMPT = """You are an expert IT Helpdesk Assistant for a company.
Help employees resolve IT issues quickly and accurately.

RULES:
1. Always use search_knowledge_base first for any IT issue.
2. Also use search_past_tickets if issue sounds recurring.
3. Provide step-by-step resolution instructions.
4. Mention which department/tags the answer came from.
5. NEVER auto-escalate — always ask user permission first.
6. Only call escalate_issue tool AFTER user confirms with yes.
7. Before escalating, ask for user email address.
8. Never make up solutions — only use what tools return.

ESCALATION FLOW:
- If no solution found → ask: Should I escalate this to a human IT agent? (yes/no)
- If user says yes → ask: Please provide your email address so the IT agent can contact you.
- After getting email → call escalate_issue tool with issue + email included.
- If user says no → suggest they try again with more details.

FORMAT:
 Issue understood: [restate briefly]
 Source: [department + tags]
 Steps to resolve: [numbered steps]
 If this doesn't work: Should I escalate this to a human IT agent? (yes/no)
"""

    tools = [search_knowledge_base, search_past_tickets, escalate_issue]
    return create_react_agent(
        llm,
        tools,
        prompt=SystemMessage(content=SYSTEM_PROMPT)
    )


st.set_page_config(
    page_title="IT Helpdesk Assistant",
    page_icon="🖥️",
    layout="centered"
)
st.title("🖥️ IT Helpdesk")
st.caption("Made By Muhammad Rabees")

# ── Chat history
if "messages" not in st.session_state:
    st.session_state.messages = [{
        "role": "assistant",
        "content": "👋 Hi! I am your IT Helpdesk Assistant. Describe your IT issue."
    }]

# ── Display chat history ──────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Describe your IT issue..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner(" Wait While the agent is searching for a solution..."):
            agent = load_agent()

            history = []
            for msg in st.session_state.messages[:-1]:
                role = "human" if msg["role"] == "user" else "assistant"
                history.append((role, msg["content"]))
            history.append(("human", prompt))

            result = agent.invoke({"messages": history})

            answer = ""
            for msg in reversed(result["messages"]):
                if hasattr(msg, "type") and msg.type == "ai" and msg.content:
                    content = msg.content
                    if isinstance(content, list):
                        answer = "\\n".join(
                            item["text"] for item in content
                            if isinstance(item, dict) and item.get("type") == "text"
                        )
                    else:
                        answer = content
                    break

            if not answer:
                answer = "I could not find a solution. Should I escalate this to a human IT agent? (yes/no)"

        st.markdown(answer)
        st.session_state.messages.append({"role": "assistant", "content": answer})

with st.sidebar:
    if st.button(" Clear chat"):
        st.session_state.messages = []
        st.rerun()
'''

with open("/content/app.py", "w") as f:
    f.write(STREAMLIT_APP)

print(" app.py written!")

Key found: gsk_kI3v...
 app.py written!


**Launching the app**

In [9]:

!pip install -q pyngrok

!streamlit run /content/app.py &>/content/streamlit.log &

from pyngrok import ngrok
from google.colab import userdata
import time

time.sleep(3)

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

public_url = ngrok.connect(8501)
print(f" Chatbot live at: {public_url}")

 Chatbot live at: NgrokTunnel: "https://march-speller-headset.ngrok-free.dev" -> "http://localhost:8501"
